In [ ]:
import tkinter 
from tkinter import ttk
import sv_ttk
import csv
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.figure import Figure 
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from PyAstronomy import pyasl
import matplotlib.animation as animation
from sgp4.api import Satrec, jday

root = tkinter.Tk()
root.title("Satellite Trajectory Analysis")
root.geometry("1200x800")

sv_ttk.set_theme("dark")
plt.style.use('dark_background')
style = ttk.Style()
style.configure('Margin.TLabel', padding=(0, 20, 0, 5))
style.configure('TButton', padding=(10, 10), background="#000")

left_frame = tkinter.Frame(root, width=500, height=600)
right_frame = tkinter.Frame(root, width=300, height=600)
left_frame.pack(side="left", fill="both", expand=True)
right_frame.pack(side="left", fill="both", expand=True)

semi_major_axis_label = ttk.Label(right_frame, text="Semi-Major Axis: ", style='Margin.TLabel')
semi_major_axis_label.pack()
semi_major_axis_slider = tkinter.Scale(right_frame, from_=2000, to=50000, tickinterval=10000, orient=tkinter.HORIZONTAL, length=400)
semi_major_axis_slider.pack()

eccentricity_label = ttk.Label(right_frame, text="Eccentricity: ", style='Margin.TLabel')
eccentricity_label.pack()
eccentricity_slider = tkinter.Scale(right_frame, from_=0, to=1, resolution=0.01, tickinterval=0.1, orient=tkinter.HORIZONTAL, length=400)
eccentricity_slider.pack()

inclination_label = ttk.Label(right_frame, text="Inclination: ", style='Margin.TLabel')
inclination_label.pack()
inclination_slider = tkinter.Scale(right_frame, from_=0, to=360, tickinterval=40, orient=tkinter.HORIZONTAL, length=400)
inclination_slider.pack()

raan_label = ttk.Label(right_frame, text="Right Ascension of Ascending Node (RAAN): ", style='Margin.TLabel')
raan_label.pack()
raan_slider = tkinter.Scale(right_frame, from_=0, to=360, tickinterval=40, orient=tkinter.HORIZONTAL, length=400)
raan_slider.pack()

argument_periapsis_label = ttk.Label(right_frame, text="Argument of Perigee: ", style='Margin.TLabel')
argument_periapsis_label.pack()
argument_periapsis_slider = tkinter.Scale(right_frame, from_=1, to=360, tickinterval=40, orient=tkinter.HORIZONTAL, length=400)
argument_periapsis_slider.pack()

mean_anamoly_label = ttk.Label(right_frame, text="Mean Anomaly: ", style='Margin.TLabel')
mean_anamoly_label.pack()
mean_anamoly_slider = tkinter.Scale(right_frame, from_=1, to=360, tickinterval=40, orient=tkinter.HORIZONTAL, length=400)
mean_anamoly_slider.pack()

def display_sliders(a, e, i, o, w, v):
    semi_major_axis_slider.set(a)
    eccentricity_slider.set(e)
    inclination_slider.set(i)
    raan_slider.set(o)
    argument_periapsis_slider.set(w)
    mean_anamoly_slider.set(v)

display_sliders(10000, 0.1, 90, 40, 1, 1)

def trace_altitude_graph(tle_one, tle_two):
    satellite = Satrec.twoline2rv(tle_one, tle_two)
    start_time = 0
    end_time = 24 * 3600
    step = 60
    times = np.arange(start_time, end_time, step)
    altitudes = []
    for t in times:
        jd, fr = jday(2024, 4, 1, 0, 0, t)
        e, r, v = satellite.sgp4(jd, fr)
        altitude = (r[0]**2 + r[1]**2 + r[2]**2)**0.5 - 6378.135
        altitudes.append(altitude)
    fig = Figure(figsize=(6, 3), dpi=100) 
    canvas = FigureCanvasTkAgg(fig, master=left_frame) 
    canvas.get_tk_widget().pack(pady=15)
    plot = fig.add_subplot(111)
    plot.plot(times, altitudes)
    plot.grid(True)
    canvas.draw() 

trace_altitude_graph(
    "1 25544U 98067A   21257.91276829  .00000825  00000-0  24323-4 0  9990", 
    "2 25544  51.6461  89.6503 0003031 120.4862 259.0942 15.4888108230711"
)

fig2 = Figure(figsize=(6, 3), dpi=100) 
canvas2 = FigureCanvasTkAgg(fig2, master=left_frame) 

def visualize_3d_orbit(a, p, e, o, i, w, fig_vis):
    orbit = pyasl.KeplerEllipse(a=a, per=p, e=e, Omega=o, i=i, w=w)
    t = np.linspace(0, 4, 300)
    pos = orbit.xyzPos(t)
    if fig_vis:
        fig_vis.clear()
    canvas2.get_tk_widget().pack(pady=15)
    plot2 = fig_vis.add_subplot(111, projection='3d')
    plot2.plot(0, 0, 'bo', markersize=9, label="Earth")
    plot2.plot(pos[::, 1], pos[::, 0], 'k-', label="Satellite Trajectory")
    plot2.plot(pos[0, 1], pos[0, 0], 'g*', label="Periapsis")
    canvas2.draw() 

visualize_3d_orbit(1.0, 1.0, 0.5, 0.0, 30.0, 0.0, fig2)

def retrace_orbit():
    a = semi_major_axis_slider.get()
    e = eccentricity_slider.get()
    i = inclination_slider.get()
    o = raan_slider.get()
    w = argument_periapsis_slider.get()
    v = mean_anamoly_slider.get()
    visualize_3d_orbit(a, v, e, o, i, w, fig2)

retrace_button = ttk.Button(right_frame, text="Retrace Orbit", style='TButton', command=retrace_orbit)
retrace_button.pack(pady=10)

def download_csv():
    with open("data.csv", mode="w", newline="") as file:
        fieldnames = ["A", "E", "I", "O", "W", "V"]
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerow({
            "A": semi_major_axis_slider.get(),
            "E": eccentricity_slider.get(),
            "I": inclination_slider.get(),
            "O": raan_slider.get(),
            "W": argument_periapsis_slider.get(),
            "V": mean_anamoly_slider.get()
        })

download_button = ttk.Button(right_frame, text="Download CSV", style='TButton', command=download_csv)
download_button.pack(pady=10)

root.mainloop()
